# Account-Level Loss Compression — lite

Compresses an account-level loss table, reconstructs the per-account metrics, and reports marginal
impact at a return period.

**Two possible inputs.** Set `INPUT_MODE` in the config cell:

| mode | you provide | §0 does |
|---|---|---|
| `"PLT"` | `account_plt`, `portfolio_plt` | nothing — reads them straight |
| `"ELT"` | `elt` (account-level) | simulates both PLTs from it |

Everything from §1 onwards is identical either way.

| table | columns |
|---|---|
| `elt` | `event_id, accnt_no, rate, mean_loss, sd_indep, sd_corr, exposure` |
| `account_plt` | `accnt_no, event_id, year_id, loss_date, loss` |
| `portfolio_plt` | `event_id, year_id, loss_date, loss` |

Both bases are handled: **AEP** (annual sum) and **OEP** (largest single occurrence). They need
different reconstruction paths, which is most of what §4 is about.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats, sparse

Q_RETAIN  = 0.98      # store the worst 2% of years exactly
ALPHA     = 0.99      # metrics reported at this level
BLOCK     = 50_000    # occurrences per block in the OEP scan (sets peak memory)
OCC_KEY   = ["year_id", "event_id", "loss_date"]

INPUT_MODE = "ELT"     # "ELT" -> simulate the PLTs from an account ELT
                       # "PLT" -> read two PLTs directly, skip section 0
T_YEARS    = 20_000    # trial years (ELT mode; in PLT mode it is inferred)
SEED       = 42        # engine seed; the same seed reproduces the tables exactly

assert ALPHA >= Q_RETAIN, "a metric level outside the retained region is not covered"
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

## 0 · ELT input — simulate the PLTs

Skip this section entirely if you already have the two PLTs (`INPUT_MODE = "PLT"`).

### Why the portfolio ELT→YLT algorithm cannot be reused

| | portfolio grain | account grain |
|---|---|---|
| one ELT row is | one event | one `(event, account)` pair |
| frequency | Poisson superposition over rows, then sample which row | Poisson per **event**, then loop accounts within |
| severity | `rng.beta(a, b)` per row, independent | `beta.ppf(norm.cdf(shared + private))` |
| `sd_corr` | folded into `sd_tot`, treated as independent | drives a shock **shared** across the footprint |

Feeding an account ELT to the portfolio algorithm fails *silently*: pooling `rate` across
`(event, account)` rows inflates frequency by roughly the accounts-per-event count, and sampling one
row per occurrence makes accounts independent — one hurricane hitting eight accounts becomes eight
separate occurrences hitting one account each. The portfolio tail collapses and every diversification
and marginal-impact number below is wrong, with no error raised.

### Severity convention

    sd_tot = sd_indep + sd_corr                       (linear, RMS-style)
    damage ratio ~ Beta(mean = mean_loss/exposure, sd = sd_tot/exposure)
    loss = damage ratio x exposure

Correlation comes from a Gaussian copula:

    u    = Phi( w_corr * z_shared + w_indep * z_private )
    loss = BetaPPF(u; a, b) * exposure

with `w_corr, w_indep` proportional to `sd_corr, sd_indep` and normalised so
`w_corr^2 + w_indep^2 = 1`. The Beta **marginal** is identical to `rng.beta(a, b)` — a quantile
transform of a uniform *is* a Beta. Only the dependence changes.

In [4]:
REQUIRED = {"event_id", "accnt_no", "rate", "mean_loss",
            "sd_indep", "sd_corr", "exposure"}

# --- the engine (only used when INPUT_MODE == 'ELT') ---

def beta_params_from_moments(m, s):
    """Method-of-moments Beta from mean ratio `m` and sd ratio `s`, both in (0, 1).
    Returns (a, b, capped)."""
    m = np.asarray(m, dtype=float)
    s = np.asarray(s, dtype=float)
    s_max  = np.sqrt(m * (1.0 - m))
    capped = s >= s_max
    s  = np.where(capped, np.nextafter(s_max, 0.0) * (1 - 1e-9), s)
    nu = m * (1.0 - m) / s ** 2 - 1.0
    return m * nu, (1.0 - m) * nu, capped

def elt_to_account_plt(elt, n_years, seed=42, secondary=True, days_in_year=365,
                       drop_zero=True, zero_tol=0.0, verbose=True):
    """Simulate an account-level PLT from an account-level ELT.

    Parameters
    ----------
    elt          : DataFrame with the REQUIRED columns, one row per (event_id, accnt_no)
    n_years      : number of trial years
    seed         : root seed; the same seed reproduces the table bit for bit
    secondary    : if False, use the deterministic mean_loss (accounts still co-occur)
    days_in_year : calendar length used to place occurrences on dates
    drop_zero    : drop rows with loss <= zero_tol, mirroring a vendor extract that does
                   not write sub-threshold losses; set zero_tol to your real threshold
    verbose      : print the Beta-capping warning if any occurrence hits the bound

    Returns
    -------
    DataFrame: accnt_no, event_id, year_id (1-based), loss_date, loss
    """
    missing = REQUIRED - set(elt.columns)
    if missing:
        raise ValueError(f"account ELT missing columns: {sorted(missing)}")
    if n_years < 1:
        raise ValueError("n_years must be >= 1")

    # --- rate is a property of the EVENT, not of the (event, account) pair ---
    rbe = elt.groupby("event_id")["rate"].agg(["min", "max", "first"])
    if not np.allclose(rbe["min"], rbe["max"]):
        bad = rbe.index[~np.isclose(rbe["min"], rbe["max"])].tolist()
        raise ValueError(f"rate varies within event_id for events {bad[:5]}"
                         f"{' ...' if len(bad) > 5 else ''}; it must be constant across accounts")

    rng = np.random.default_rng(seed)

    # stable order == reproducible RNG consumption
    elt    = elt.sort_values(["event_id", "accnt_no"], kind="stable")
    events = rbe.index.to_numpy()
    rates  = rbe["first"].to_numpy(dtype=float)

    frames, n_capped = [], 0

    for event_id, rate in zip(events, rates):
        legs = elt[elt["event_id"] == event_id]

        # --- 1. frequency: Poisson per EVENT, not per row -------------------
        counts = rng.poisson(rate, size=n_years)
        n_occ  = int(counts.sum())
        if n_occ == 0:
            continue
        year = np.repeat(np.arange(1, n_years + 1), counts)

        # --- 2. dates distinct within (year, event) -------------------------
        start  = np.concatenate([[0], np.cumsum(counts)])
        within = np.arange(n_occ) - np.repeat(start[:-1], counts)
        base   = rng.integers(1, days_in_year + 1, size=n_years)
        day    = 1 + (base[year - 1] - 1 + within) % days_in_year

        # --- 3. THE COMMON SHOCK: one draw per occurrence, shared by every --
        #        account exposed to it. This is the whole mechanism.
        z_shared = rng.standard_normal(n_occ) if secondary else None

        # --- 4. per account in the footprint --------------------------------
        for leg in legs.itertuples(index=False):
            mu, E = float(leg.mean_loss), float(leg.exposure)

            if not secondary or E <= 0:
                loss = np.full(n_occ, mu, dtype=float)
            else:
                m = mu / E
                s = (float(leg.sd_indep) + float(leg.sd_corr)) / E
                if s <= 0 or m <= 0 or m >= 1:
                    loss = np.full(n_occ, mu, dtype=float)          # degenerate -> mean
                else:
                    a, b, capped = beta_params_from_moments(m, s)
                    n_capped += int(np.atleast_1d(capped).sum())

                    w = np.hypot(float(leg.sd_corr), float(leg.sd_indep))
                    if w <= 0:
                        w_corr, w_indep = 0.0, 1.0
                    else:
                        w_corr, w_indep = float(leg.sd_corr) / w, float(leg.sd_indep) / w

                    z    = w_corr * z_shared + w_indep * rng.standard_normal(n_occ)
                    loss = stats.beta.ppf(stats.norm.cdf(z), a, b) * E

            frames.append(pd.DataFrame({"accnt_no": leg.accnt_no, "event_id": event_id,
                                        "year_id": year, "loss_date": day, "loss": loss}))

    if not frames:
        return pd.DataFrame(columns=["accnt_no", "event_id", "year_id", "loss_date", "loss"])

    out = pd.concat(frames, ignore_index=True)
    if drop_zero:
        out = out[out["loss"] > zero_tol].reset_index(drop=True)
    if verbose and n_capped:
        print(f"[warn] sd capped at the Beta bound for {n_capped} account-events")

    return (out.sort_values(["year_id", "event_id", "loss_date", "accnt_no"])
               .reset_index(drop=True))

def aggregate_to_portfolio_plt(account_plt):
    """Sum away the account dimension: one row per occurrence."""
    return (account_plt.groupby(["year_id", "event_id", "loss_date"], as_index=False)["loss"]
            .sum().sort_values(["year_id", "event_id", "loss_date"]).reset_index(drop=True))


def validate_account_plt(account_plt, elt, n_years, z_tol=4.0):
    """Per-account AAL against the ELT analytic value, in units of Monte Carlo error."""
    analytic = elt.assign(_a=elt["rate"] * elt["mean_loss"]).groupby("accnt_no")["_a"].sum()
    var = (elt.assign(_v=elt["rate"] * (elt["mean_loss"] ** 2
                                        + (elt["sd_indep"] + elt["sd_corr"]) ** 2))
              .groupby("accnt_no")["_v"].sum())
    sim = (account_plt.groupby("accnt_no")["loss"].sum()
           .reindex(analytic.index).fillna(0.0) / n_years)
    se  = np.sqrt(var / n_years)
    res = pd.DataFrame({"aal_analytic": analytic, "aal_simulated": sim,
                        "se": se, "z": (sim - analytic) / se})
    res["ok"] = res["z"].abs() < z_tol
    return res

### The ELT

Replace this cell with your own load. If your columns are named differently (`EventId`,
`AccountId`, `MeanLoss`, …) rename them first — `itertuples` accesses by attribute, so
`leg.mean_loss` would otherwise need to become `leg.MeanLoss`.

In [5]:
if INPUT_MODE == "ELT":
    _r0      = np.random.default_rng(7)
    _n_ev    = 30
    accounts = [f"ACC-{i:02d}" for i in range(1, 9)]
    _expo    = dict(zip(accounts, [2500, 1800, 1200, 900, 3000, 600, 1500, 2200]))
    _rates   = dict(zip(range(1, _n_ev + 1), 0.30 * 0.88 ** np.arange(_n_ev)))

    _recs = []
    for a in accounts:
        _hit = np.arange(1, _n_ev + 1)[_r0.random(_n_ev) < 0.45]        # this account's footprint
        for e, m in zip(_hit, np.geomspace(0.002, _r0.uniform(0.05, 0.30), len(_hit))):
            ml = m * _expo[a]
            _recs.append({"event_id": e, "accnt_no": a, "rate": _rates[e], "mean_loss": ml,
                          "sd_indep": 0.40 * ml, "sd_corr": 0.30 * ml, "exposure": float(_expo[a])})
    elt = pd.DataFrame(_recs).sort_values(["event_id", "accnt_no"]).reset_index(drop=True)

    print(f"account ELT: {len(elt)} rows | {elt.event_id.nunique()} events | "
          f"{elt.accnt_no.nunique()} accounts | "
          f"{elt.groupby('event_id').size().mean():.1f} accounts per event")
    print(f"portfolio AAL implied by the ELT = {(elt.rate * elt.mean_loss).sum():,.2f}")
    display(elt.head())

account ELT: 106 rows | 28 events | 8 accounts | 3.8 accounts per event
portfolio AAL implied by the ELT = 215.25


,event_id,accnt_no,rate,mean_loss,sd_indep,sd_corr,exposure
0,1,ACC-02,0.300,3.600,1.440,1.080,"1,800.000"
1,1,ACC-07,0.300,3.000,1.200,0.900,"1,500.000"
2,2,ACC-02,0.264,5.019,2.008,1.506,"1,800.000"
3,2,ACC-03,0.264,2.400,0.960,0.720,"1,200.000"
4,2,ACC-04,0.264,1.800,0.720,0.540,900.000


In [ ]:
# importing ELT from a database instead of simulating it
'''if INPUT_MODE == "ELT":
    import sqlalchemy as sa
    eng = sa.create_engine("mssql+pyodbc://SERVER/DB?driver=ODBC+Driver+17+for+SQL+Server")

    elt = pd.read_sql("""
        SELECT EventId    AS event_id,
               AccountId  AS accnt_no,
               Rate       AS rate,
               MeanLoss   AS mean_loss,
               StdDevI    AS sd_indep,
               StdDevC    AS sd_corr,
               Exposure   AS exposure
        FROM   dbo.AccountELT
        WHERE  AnalysisId = ?
    """, eng, params=(analysis_id,))

    accounts = sorted(elt.accnt_no.unique())
    display(elt.head())'''

### Simulate

`year_id` is shifted to 0-based because everything downstream indexes arrays of length `T_YEARS`
directly.

In [6]:
if INPUT_MODE == "ELT":
    account_plt = elt_to_account_plt(elt, n_years=T_YEARS, seed=SEED)
    account_plt["year_id"] -= 1                       # engine is 1-based; arrays are 0-based
    portfolio_plt = (account_plt.groupby(OCC_KEY, as_index=False)["loss"].sum())
    print(f"simulated {len(account_plt):,} account rows -> {len(portfolio_plt):,} occurrences")

simulated 206,737 account rows -> 47,027 occurrences


### Engine checks

Run these once on a new ELT. The last one is what matters — the others can all pass while the engine
is quietly producing independent accounts.

In [7]:
if INPUT_MODE == "ELT":
    # 1. AAL closes against the ELT (z-scores, not flat percentages)
    display(validate_account_plt(account_plt, elt, T_YEARS).round(3))

    # 2. occurrence key unique -- OEP needs this
    assert not portfolio_plt.duplicated(OCC_KEY).any()
    print("occurrence key unique: True")

    # 3. same seed -> identical table
    _again = elt_to_account_plt(elt, n_years=T_YEARS, seed=SEED, verbose=False)
    assert np.array_equal(np.sort(account_plt.loss.to_numpy()), np.sort(_again.loss.to_numpy()))
    print("reproducible under the same seed: True"); del _again

    # 4. accounts MUST co-move within an event -- this is what sd_corr buys
    _ev  = portfolio_plt.event_id.value_counts().index[0]
    _piv = (account_plt[account_plt.event_id == _ev]
            .pivot_table(index=["year_id", "loss_date"], columns="accnt_no", values="loss"))
    print(f"\nwithin-event correlation of account losses, event {_ev}:")
    display(_piv.corr().round(3))
    print("If these are ~0 the shared shock is not wired through: the portfolio tail would be far")
    print("too thin and every number below would be wrong while looking reasonable.")

,aal_analytic,aal_simulated,se,z,ok
accnt_no,,,,,
ACC-01,31.893,31.316,0.635,-0.909,True
ACC-02,36.566,36.088,0.565,-0.846,True
ACC-03,17.740,17.436,0.270,-1.124,True
ACC-04,12.948,12.685,0.219,-1.201,True
ACC-05,58.210,58.011,0.918,-0.217,True
ACC-06,5.820,5.837,0.071,0.245,True
ACC-07,33.213,32.791,0.524,-0.806,True
ACC-08,18.863,18.732,0.502,-0.260,True


occurrence key unique: True
reproducible under the same seed: True

within-event correlation of account losses, event 1:


accnt_no,ACC-02,ACC-07
accnt_no,,
ACC-02,1.000,0.350
ACC-07,0.350,1.000


If these are ~0 the shared shock is not wired through: the portfolio tail would be far
too thin and every number below would be wrong while looking reasonable.


## 1 · The input tables

In `INPUT_MODE = "PLT"` this is where you read your two tables. In `"ELT"` mode they already exist
from §0 and this cell only derives the bookkeeping.

Casting `accnt_no` to `category` and the ids to `int32` roughly halves memory on a 10M-row table and
speeds up every groupby below.

In [ ]:
# importing from SQL database instead of simulating the PLT


In [ ]:
if INPUT_MODE == "PLT":
    # ---- replace with your own load -------------------------------------
    # account_plt   = pd.read_parquet("account_plt.parquet")
    # portfolio_plt = pd.read_parquet("portfolio_plt.parquet")
    
    '''
    account_plt = pd.read_sql("""
        SELECT AccountId AS accnt_no, EventId AS event_id,
               TrialId   AS year_id,  LossDate AS loss_date, Loss AS loss
        FROM   dbo.AccountPLT WHERE AnalysisId = ?
    """, eng, params=(analysis_id,))

    portfolio_plt = account_plt.groupby(OCC_KEY, as_index=False)["loss"].sum()'''
    raise NotImplementedError("point INPUT_MODE='PLT' at your own read_parquet calls")

accounts = sorted(account_plt["accnt_no"].unique())
A_IDX    = {a: i for i, a in enumerate(accounts)}
N_ACC    = len(accounts)
if INPUT_MODE == "PLT":
    # infer only when there is no metadata: an empty final year silently shrinks T
    # (P = exp(-Lambda) per run) and shifts every quantile rank downstream. Prefer the
    # analysis's trial count if the source system provides it.
    T_YEARS = int(account_plt["year_id"].max()) + 1
assert int(account_plt["year_id"].max()) < T_YEARS, "year_id exceeds T_YEARS"

assert not portfolio_plt.duplicated(OCC_KEY).any(), "occurrence key is not unique"
assert np.isclose(account_plt.loss.sum(), portfolio_plt.loss.sum()), "the two PLTs disagree"

print(f"{len(account_plt):,} account rows | {len(portfolio_plt):,} occurrences | "
      f"{N_ACC} accounts | {T_YEARS:,} years")
print(f"accounts per occurrence {len(account_plt)/len(portfolio_plt):.2f}")
display(account_plt.head())

206,737 account rows | 47,027 occurrences | 8 accounts | 20,000 years
accounts per occurrence 4.40


,accnt_no,event_id,year_id,loss_date,loss
0,ACC-02,1,0,237,2.582
1,ACC-07,1,0,237,2.415
2,ACC-02,6,0,75,13.993
3,ACC-03,6,0,75,7.483
4,ACC-04,6,0,75,5.379


## 2 · Compress

**Step 1 — the two annual metrics.** `Y[t]` = total loss in year `t` (AEP), `X[t]` = largest single
occurrence in year `t` (OEP).

**Step 2 — pick the years to keep.** The worst `(1−q)·T` by `Y`, **union** the worst `(1−q)·T` by `X`.
The union matters: a year with one enormous event ranks high on OEP but may not on AEP, and a year of
several moderate events ranks high on AEP but not OEP. Ranking is on the **year**, never on
individual occurrences — a bad year can be several moderate events, none individually large.

**Step 3 — split.** Retained years: every account row kept as-is. All other years ("the body"): rows
discarded, replaced by one share vector per event.

The share is `Σ losses to account a ÷ Σ portfolio loss` over all body occurrences of that event —
**sum first, then divide**. Averaging per-row ratios instead would bias against accounts that take a
larger slice of the larger events.

In [9]:
# --- 1. annual metrics -----------------------------------------------------
Y = np.bincount(portfolio_plt.year_id, weights=portfolio_plt.loss, minlength=T_YEARS)   # AEP
X = (portfolio_plt.groupby("year_id")["loss"].max()
     .reindex(range(T_YEARS), fill_value=0.0).to_numpy())                               # OEP

# --- 2. retained years: union of the AEP and OEP tails --------------------
k        = int(round((1 - Q_RETAIN) * T_YEARS))   # same convention as m in section 5
rank     = lambda v, n: np.argsort(-v, kind="stable")[:n]
keep_yrs = np.union1d(rank(Y, k), rank(X, k))
is_kept  = np.zeros(T_YEARS, bool); is_kept[keep_yrs] = True

# --- 3a. retained rows, verbatim ------------------------------------------
tail_store = account_plt[is_kept[account_plt.year_id.to_numpy()]].copy()

# --- 3b. body rows -> one share vector per event --------------------------
body_acct = account_plt[~is_kept[account_plt.year_id.to_numpy()]]
body_port = portfolio_plt[~is_kept[portfolio_plt.year_id.to_numpy()]]

numer  = body_acct.groupby(["event_id", "accnt_no"])["loss"].sum()      # sum first ...
denom  = body_port.groupby("event_id")["loss"].sum()
shares = (numer / denom).unstack(fill_value=0.0).reindex(columns=accounts).fillna(0.0)  # ... then divide

assert np.allclose(shares.sum(axis=1), 1.0), "each event's shares must sum to 1"
print(f"AEP tail {k:,} years | OEP tail {k:,} | union {len(keep_yrs):,} "
      f"({len(keep_yrs)/k:.2f}x a single metric)")
print(f"stored {len(tail_store):,} rows = {100*len(tail_store)/len(account_plt):.1f}% of the "
      f"account table, plus a {shares.shape[0]} x {shares.shape[1]} share table")
display(shares.head())

AEP tail 401 years | OEP tail 401 | union 461 (1.15x a single metric)
stored 7,199 rows = 3.5% of the account table, plus a 28 x 8 share table


accnt_no,ACC-01,ACC-02,ACC-03,ACC-04,ACC-05,ACC-06,ACC-07,ACC-08
event_id,,,,,,,,
1,0.000,0.546,0.000,0.000,0.000,0.000,0.454,0.000
2,0.000,0.201,0.095,0.072,0.240,0.048,0.167,0.176
3,0.000,0.266,0.124,0.000,0.333,0.063,0.214,0.000
4,0.140,0.000,0.126,0.073,0.367,0.066,0.227,0.000
5,0.115,0.144,0.092,0.056,0.283,0.050,0.161,0.100


## 3 · What reconstruction means

For a retained year, read the stored row. For a body occurrence with event `e` and portfolio loss
`L`:

    loss(account a) = share[e, a] × L

Note this is *denser* than the truth: the share vector is non-zero for every account ever touched by
event `e`, so reconstruction assigns a loss to all of them on every occurrence, where the real table
only has rows for the accounts actually hit that time. Check the inflation on your own data before
materialising anything row-level:

In [10]:
per_occ   = len(account_plt) / len(portfolio_plt)
per_event = account_plt.groupby("event_id")["accnt_no"].nunique().mean()
print(f"accounts per occurrence {per_occ:.2f} | accounts per event {per_event:.2f} "
      f"-> row inflation {per_event/per_occ:.2f}x")
print(f"a row-level reconstruction would be ~{len(portfolio_plt)*per_event:,.0f} rows "
      f"vs {len(account_plt):,} true")
print("\nThe metrics below need per-account ANNUAL values, so the row-level table is never built.")

accounts per occurrence 4.40 | accounts per event 3.79 -> row inflation 0.86x
a row-level reconstruction would be ~178,031 rows vs 206,737 true

The metrics below need per-account ANNUAL values, so the row-level table is never built.


## 4 · Reconstruct the annual metrics

The two bases need different routes, and this is the part that matters at scale.

**AEP — annual sum.** Summing over occurrences is exactly a matrix product: group body losses by
`(year, event)`, then multiply that sparse matrix by the share table. Output is `T × n_accounts`
directly; nothing occurrence-level is ever materialised.

**OEP — annual max.** A max cannot be recovered from sums, so per-occurrence values are needed. But
they are consumed in **blocks**: build `BLOCK × n_accounts` at a time and fold it into a running max.
Peak memory is `BLOCK × n_accounts × 8` bytes no matter how many occurrences there are — 200 MB at
50,000 × 500. This is the one place blocking is genuinely required.

Retained years are added afterwards straight from the stored rows, so they stay exact.

In [11]:
# ---------- AEP: annual SUM per account -----------------------------------
AEP = np.zeros((T_YEARS, N_ACC))
by  = body_port.groupby(["year_id", "event_id"], observed=True)["loss"].sum().reset_index()
ec  = shares.index.get_indexer(by.event_id)
assert (ec >= 0).all(), "a body occurrence has an event absent from the share table"
S   = sparse.csr_matrix((by.loss.to_numpy(), (by.year_id.to_numpy(), ec)),
                        shape=(T_YEARS, len(shares)))
AEP += S @ shares.to_numpy()                                     # body, in one product
np.add.at(AEP, (tail_store.year_id.to_numpy(), tail_store.accnt_no.map(A_IDX).to_numpy()),
          tail_store.loss.to_numpy())                            # retained years, exact

# ---------- OEP: annual MAX per account -----------------------------------
OEP  = np.zeros((T_YEARS, N_ACC))
Sarr = shares.to_numpy()
ecb  = shares.index.get_indexer(body_port.event_id)
yrb, lsb = body_port.year_id.to_numpy(), body_port.loss.to_numpy()
for s in range(0, len(body_port), BLOCK):
    sl = slice(s, s + BLOCK)
    np.maximum.at(OEP, yrb[sl], Sarr[ecb[sl]] * lsb[sl][:, None])   # only BLOCK x N_ACC alive
np.maximum.at(OEP, (tail_store.year_id.to_numpy(), tail_store.accnt_no.map(A_IDX).to_numpy()),
              tail_store.loss.to_numpy())                            # retained years, exact

print(f"AEP via sparse product | OEP via {len(range(0, len(body_port), BLOCK))} blocks of "
      f"{BLOCK:,} (peak {BLOCK*N_ACC*8/1e6:.0f} MB)")

AEP via sparse product | OEP via 1 blocks of 50,000 (peak 3 MB)


In [12]:
# ---------- the truth, for comparison only --------------------------------
ay, aa, al = (account_plt.year_id.to_numpy(), account_plt.accnt_no.map(A_IDX).to_numpy(),
              account_plt.loss.to_numpy())
AEP_T = np.zeros((T_YEARS, N_ACC)); np.add.at(AEP_T, (ay, aa), al)
OEP_T = np.zeros((T_YEARS, N_ACC)); np.maximum.at(OEP_T, (ay, aa), al)
print(f"AEP totals reconcile: {np.allclose(AEP_T.sum(1), Y)} | "
      f"OEP portfolio max reconciles: {np.allclose(portfolio_plt.groupby('year_id').loss.max().reindex(range(T_YEARS), fill_value=0), X)}")

AEP totals reconcile: True | OEP portfolio max reconciles: True


## 5 · The metrics

- **AAL** — mean annual loss
- **VaR / TVaR** at 99%, standalone: on the account's *own* worst years
- **co-TVaR** — the allocation metric, read in the years where the **portfolio** is worst

co-TVaR is defined differently on the two bases, and the OEP one matters:

- *AEP basis*: the account's annual sum, averaged over the portfolio's worst `Y` years.
- *OEP basis*: the account's loss **on the single largest portfolio occurrence** of each worst `X`
  year — not the account's own annual max. Taking each account's own max would not sum back to the
  portfolio OEP TVaR, because different accounts peak on different occurrences. Reading the one
  occurrence keeps the allocation additive.

In [13]:
m         = int(round((1 - ALPHA) * T_YEARS))
aep_tail  = rank(Y, m)                          # portfolio's worst years, AEP basis
oep_tail  = rank(X, m)                          # ... OEP basis

# account losses ON each year's single biggest portfolio occurrence, for EVERY year:
# BIG from the store (retained years exact, body years via shares), BIG_T from the true
# table for comparison. Section 5 reads the OEP-tail rows; section 9 reads the windows.
big     = portfolio_plt.loc[portfolio_plt.groupby("year_id")["loss"].idxmax()]
big_key = pd.MultiIndex.from_frame(big[OCC_KEY])

def big_matrix(rows):
    hit = rows.set_index(OCC_KEY)
    hit = hit[hit.index.isin(big_key)].reset_index()
    M = np.zeros((T_YEARS, N_ACC))
    M[hit.year_id.to_numpy(), hit.accnt_no.map(A_IDX).to_numpy()] = hit.loss.to_numpy()
    return M

BIG_T = big_matrix(account_plt)                      # truth (comparison only)
BIG   = big_matrix(tail_store)                       # retained years, exact
bb    = big[~is_kept[big.year_id.to_numpy()]]        # body years: shares x occurrence loss
BIG[bb.year_id.to_numpy()] = (shares.to_numpy()[shares.index.get_indexer(bb.event_id)]
                              * bb.loss.to_numpy()[:, None])
rec_occ, true_occ = BIG[oep_tail].mean(0), BIG_T[oep_tail].mean(0)

def table(A_, O_, occ_):
    sa, so = np.sort(A_, 0), np.sort(O_, 0)
    return pd.DataFrame({"AEP AAL": A_.mean(0),
                         "AEP VaR": sa[-m], "AEP TVaR": sa[-m:].mean(0),
                         "AEP coTVaR": A_[aep_tail].mean(0),
                         "OEP VaR": so[-m], "OEP TVaR": so[-m:].mean(0),
                         "OEP coTVaR": occ_}, index=accounts)

true, recon = table(AEP_T, OEP_T, true_occ), table(AEP, OEP, rec_occ)
err = (recon - true).abs() / true.replace(0, np.nan) * 100
display(pd.concat({"true": true, "reconstructed": recon}, axis=1)
        .swaplevel(axis=1).sort_index(axis=1).round(2))
display(err.round(4).rename(columns=lambda c: c + " err %"))

AEP AAL             AEP TVaR               AEP VaR          \
       reconstructed   true reconstructed    true reconstructed    true   
ACC-01        31.320 31.320       670.670 697.840       398.000 447.540   
ACC-02        36.090 36.090       578.850 599.790       367.000 406.790   
ACC-03        17.440 17.440       262.990 280.840       177.550 187.060   
ACC-04        12.680 12.680       214.870 233.950       148.410 150.390   
ACC-05        58.010 58.010       937.200 953.760       608.660 667.070   
ACC-06         5.840  5.840        62.640  69.590        45.380  50.190   
ACC-07        32.790 32.790       536.110 560.690       356.570 374.130   
ACC-08        18.730 18.730       548.320 577.940       323.390 334.340   

          AEP coTVaR              OEP TVaR               OEP VaR          \
       reconstructed    true reconstructed    true reconstructed    true   
ACC-01       419.720 419.720       641.280 670.000       367.080 423.270   
ACC-02       481.740 481.740       542.210 564.020       332.040 372.240   
ACC-03        78.750  78.750       243.490 261.150       162.000 170.760   
ACC-04        49.000  49.000       200.520 218.330       137.500 138.200   
ACC-05       668.140 668.140       887.590 909.400       552.040 602.100   
ACC-06        23.730  23.730        54.890  62.130        39.430  44.070   
ACC-07       361.200 361.200       498.860 523.130       325.640 347.890   
ACC-08       203.790 203.790       529.190 559.730       305.210 318.350   

          OEP coTVaR          
       reconstructed    true  
ACC-01       387.160 387.160  
ACC-02       446.780 446.780  
ACC-03        62.190  62.190  
ACC-04        28.290  28.290  
ACC-05       630.560 630.560  
ACC-06        15.080  15.080  
ACC-07       321.580 321.580  
ACC-08       161.990 161.990

,AEP AAL err %,AEP VaR err %,AEP TVaR err %,AEP coTVaR err %,OEP VaR err %,OEP TVaR err %,OEP coTVaR err %
ACC-01,0.000,11.071,3.894,0.000,13.275,4.288,0.000
ACC-02,0.000,9.782,3.492,0.000,10.800,3.866,0.000
ACC-03,0.000,5.083,6.355,0.000,5.133,6.759,0.000
ACC-04,0.000,1.319,8.152,0.000,0.506,8.154,0.000
ACC-05,0.000,8.756,1.736,0.000,8.313,2.398,0.000
ACC-06,0.000,9.599,9.984,0.000,10.534,11.653,0.000
ACC-07,0.000,4.694,4.383,0.000,6.394,4.638,0.000
ACC-08,0.000,3.276,5.125,0.000,4.128,5.456,0.000


In [14]:
print("Portfolio level")
print(f"  AEP TVaR   true {np.sort(Y)[-m:].mean():>12,.2f}   recon {np.sort(AEP.sum(1))[-m:].mean():>12,.2f}")
print(f"  OEP TVaR   true {np.sort(X)[-m:].mean():>12,.2f}   recon {X[oep_tail].mean():>12,.2f}")
print(f"  sum AEP coTVaR  true {true['AEP coTVaR'].sum():>10,.2f}   "
      f"recon {recon['AEP coTVaR'].sum():>10,.2f}   (= portfolio AEP TVaR)")
print(f"  sum OEP coTVaR  true {true['OEP coTVaR'].sum():>10,.2f}   "
      f"recon {recon['OEP coTVaR'].sum():>10,.2f}   (= portfolio OEP TVaR)")

print("\nWorst per-account error")
for c in err.columns:
    print(f"  {c:12s} {err[c].max():8.4f}%")

Portfolio level
  AEP TVaR   true     2,286.07   recon     2,286.07
  OEP TVaR   true     2,053.64   recon     2,053.64
  sum AEP coTVaR  true   2,286.07   recon   2,286.07   (= portfolio AEP TVaR)
  sum OEP coTVaR  true   2,053.64   recon   2,053.64   (= portfolio OEP TVaR)

Worst per-account error
  AEP AAL        0.0000%
  AEP VaR       11.0706%
  AEP TVaR       9.9838%
  AEP coTVaR     0.0000%
  OEP VaR       13.2752%
  OEP TVaR      11.6529%
  OEP coTVaR     0.0000%


## What survives

| metric | status | why |
|---|---|---|
| **AAL** | exact | shares fitted by summing then dividing, so each account's body total is reproduced identically |
| **AEP co-TVaR** | exact | reads only the portfolio's worst `Y` years, which are stored verbatim |
| **OEP co-TVaR** | exact | reads only the largest occurrence of each worst `X` year, also stored |
| **portfolio AEP / OEP** | exact | the portfolio table is kept whole |
| **per-account VaR / TVaR** | approximate | an account's own worst year is often a quiet year for the portfolio — a body year, where its loss is a share of the total rather than its real loss |

That last row is the trade. The compression is built for capital allocation, which reads the
portfolio's tail — not for per-account return periods, which read each account's own tail.

**OEP degrades more than AEP.** The share vector throws away how an event's split varies from one
occurrence to the next. A sum averages that noise out; a max is drawn to the occurrence where the
real split happened to favour an account, and that is exactly the variation the reconstruction has
flattened.

Three rules that keep it honest:

- `ALPHA >= Q_RETAIN`. A metric level outside the retained region is not covered.
- Rank on the **year**, not on individual occurrences.
- If OEP is reported, retain the **union** of the AEP and OEP tails.

## 6 · Marginal impact at a return period

Capital is usually set at a **return period** — 1-in-200 means `alpha = 1 - 1/200 = 0.995`. The
marginal impact of an account is its share of that capital.

Two ways to measure it, and they behave very differently:

**co-TVaR** — the account's average loss over the worst `m` years. Robust, because it averages many
years, and it sums exactly to portfolio TVaR.

**co-VaR** — the account's loss in the *single* year sitting at the VaR rank. On a finite table this
is one row. An account that happened not to lose in that one year gets allocated **zero capital**,
and the whole split reshuffles at the next model refresh. Standard errors below run to 90%+.

**So: use TVaR to decide the split, use VaR for the total.**

    marginal = co-TVaR proportions x portfolio VaR

It still sums exactly to VaR, but inherits co-TVaR's stability. This is standard practice for a
Solvency II style 99.5% VaR capital measure, not a workaround — VaR's own gradient conditions on a
probability-zero event, so a TVaR surrogate rescaled to the VaR total is the structurally correct
response.

The VaR year itself is inside the retained set (`ALPHA_RP >= Q_RETAIN`), so everything here is exact
under the compression — the noise is sampling noise in the underlying table, not reconstruction
error.

In [15]:
RP        = 200                      # return period
ALPHA_RP  = 1 - 1 / RP
m_rp      = int(round((1 - ALPHA_RP) * T_YEARS))
assert ALPHA_RP >= Q_RETAIN, f"RP {RP} needs alpha {ALPHA_RP}; retention only covers {Q_RETAIN}"

ord_Y   = np.argsort(-Y, kind="stable")
var_yr  = ord_Y[m_rp - 1]            # the SINGLE year sitting at the VaR rank
VaR_p   = Y[var_yr]
TVaR_p  = Y[ord_Y[:m_rp]].mean()

co_tvar  = AEP[ord_Y[:m_rp]].mean(0)          # average over the worst m years  -> robust
co_var   = AEP[var_yr]                        # that one year only              -> noisy
marginal = co_tvar / co_tvar.sum() * VaR_p    # TVaR split, VaR total           -> USE THIS

print(f"RP {RP} -> alpha {ALPHA_RP} | tail = {m_rp} of {T_YEARS:,} years")
print(f"portfolio VaR  {VaR_p:,.2f}   (year at rank {m_rp}, retained: {bool(is_kept[var_yr])})")
print(f"portfolio TVaR {TVaR_p:,.2f}")

RP 200 -> alpha 0.995 | tail = 100 of 20,000 years
portfolio VaR  2,174.31   (year at rank 100, retained: True)
portfolio TVaR 2,648.27


In [16]:
# --- bootstrap: resample years, recompute, report the spread ---------------
rb, B = np.random.default_rng(5), 400
bs_t, bs_v = np.empty((B, N_ACC)), np.empty((B, N_ACC))
for b in range(B):
    idx = rb.integers(0, T_YEARS, T_YEARS)
    Yb, Ab = Y[idx], AEP[idx]
    ob = np.argsort(-Yb, kind="stable")
    p       = Ab[ob[:m_rp]].mean(0)
    bs_t[b] = p / p.sum() * VaR_p          # the statistic itself: share x VaR (as in 7/9)
    bs_v[b] = Ab[ob[m_rp - 1]]
se_marg = bs_t.std(0)
se_cov  = bs_v.std(0)

res = pd.DataFrame({
    "MARGINAL (use this)": marginal,
    "SE":                  se_marg,
    "SE %":                100 * se_marg / marginal,
    "capital share %":     100 * marginal / marginal.sum(),
    "co-VaR (1 year)":     co_var,
    "co-VaR SE %":         100 * se_cov / np.where(co_var > 0, co_var, np.nan),
}, index=accounts)
display(res.round(2))

print(f"sum of MARGINAL {marginal.sum():,.2f} = portfolio VaR {VaR_p:,.2f}  "
      f"(diff {abs(marginal.sum()-VaR_p):.2e})")
print(f"median SE:  MARGINAL {np.nanmedian(res['SE %']):.0f}%   "
      f"raw co-VaR {np.nanmedian(res['co-VaR SE %']):.0f}%")
n0 = int((co_var == 0).sum())
if n0:
    print(f"\n{n0} of {N_ACC} accounts get ZERO capital under raw co-VaR - they simply did not lose")
    print("in that one year. That is the reason for rescaling rather than reading VaR directly.")

,MARGINAL (use this),SE,SE %,capital share %,co-VaR (1 year),co-VaR SE %
ACC-01,355.440,34.980,9.840,16.350,783.470,50.730
ACC-02,477.050,30.730,6.440,21.940,373.450,51.790
ACC-03,79.640,9.360,11.750,3.660,0.000,NaN
ACC-04,44.240,6.040,13.650,2.030,0.000,NaN
ACC-05,670.490,54.120,8.070,30.840,786.500,55.010
ACC-06,20.880,3.080,14.740,0.960,0.000,NaN
ACC-07,354.340,24.870,7.020,16.300,97.840,275.740
ACC-08,172.230,24.080,13.980,7.920,133.040,212.800


sum of MARGINAL 2,174.31 = portfolio VaR 2,174.31  (diff 4.55e-13)
median SE:  MARGINAL 11%   raw co-VaR 55%

3 of 8 accounts get ZERO capital under raw co-VaR - they simply did not lose
in that one year. That is the reason for rescaling rather than reading VaR directly.


### Reading this

`MARGINAL` is the number to price and allocate off. It sums exactly to portfolio VaR and its
bootstrap SE is a fraction of raw co-VaR's.

**Watch the SEs even so.** At RP 200 the tail is `T/200` years — 100 here, but only 50 if you run
10,000 trials. A small or diversifying account's marginal is the jumpiest figure on the page, and a
move between model runs that sits inside its own SE is not a signal. Circulate the SE beside every
allocated number.

**This is a marginal price at today's mix**, correct for "what should I charge for this account" or
"what does this renewal cost me". It is not a budget for a large move: write three times as much of a
diversifying account and it consumes its own diversification credit, so the number shifts. For
anything that changes the book materially, recompute at the target mix rather than scaling this one.

## 7 · Where the proportions come from — window vs co-TVaR

§6 took the split from **co-TVaR** (the average over every year beyond VaR) and rescaled it onto the
VaR total. A **rank window** is the alternative: average over the years *bracketing* the VaR rank —
ranks `m-K` to `m+K` — which estimates `E[L_a | S ≈ VaR]` more directly.

Both need rescaling, because a window straddles the VaR point and does not sum to VaR on its own. So
the choice is only **where the proportions come from**, and it is a bias–variance dial:

| proportions from | bias vs the true VaR gradient | noise |
|---|---|---|
| 1 year (raw co-VaR) | none — it *is* the VaR year | catastrophic |
| narrow window | small | high |
| wide window | pulled toward the body | low |
| co-TVaR | pulled toward the deep tail | lowest |

Measure both on your book rather than assuming.

In [ ]:
ord_Y = np.argsort(-Y, kind="stable")

def window_alloc(K, A=None):
    """Rescaled allocation. K=None -> co-TVaR proportions; else years bracketing the VaR rank."""
    A = AEP if A is None else A
    if K is None:
        p = A[ord_Y[:m_rp]].mean(0)
    else:
        lo, hi = max(0, m_rp - 1 - K), m_rp - 1 + K + 1
        p = A[ord_Y[lo:hi]].mean(0)
    return p / p.sum() * VaR_p

WIDTHS = [50, 200, 500, 2000]
cmp_w = pd.DataFrame({f"win{K}": window_alloc(K) for K in WIDTHS}, index=accounts)
cmp_w["co-TVaR"] = window_alloc(None)
display(cmp_w.round(0))
print("every column sums to VaR " + f"{VaR_p:,.2f}: "
      + "  ".join(f"{c} {cmp_w[c].sum():,.0f}" for c in cmp_w.columns))

In [ ]:
# --- bias (vs co-TVaR) and noise (bootstrap), per width -------------------
rel  = (cmp_w.div(cmp_w["co-TVaR"], axis=0) - 1).abs() * 100
rb, B = np.random.default_rng(5), 200
rows = []
for K in WIDTHS + [None]:
    lab = "co-TVaR" if K is None else f"win{K}"
    bs  = np.empty((B, N_ACC))
    for b in range(B):
        idx = rb.integers(0, T_YEARS, T_YEARS)
        ob  = np.argsort(-Y[idx], kind="stable"); Ab = AEP[idx]
        if K is None:
            p = Ab[ob[:m_rp]].mean(0)
        else:
            lo, hi = max(0, m_rp - 1 - K), m_rp - 1 + K + 1
            p = Ab[ob[lo:hi]].mean(0)
        bs[b] = p / p.sum() * VaR_p
    se = bs.std(0)
    rows.append({"proportions from": lab,
                 "years averaged": m_rp if K is None else min(2*K+1, T_YEARS),
                 "median diff vs co-TVaR %": float(rel[lab].median()),
                 "median SE %": float(np.nanmedian(100 * se / cmp_w[lab]))})
display(pd.DataFrame(rows).set_index("proportions from").round(2))
print("pick the narrowest width whose median SE is still tolerable; record the choice.")

## 8 · Analytic account contributions from the ELT

Everything above is **empirical**: it reads simulated years and averages. This section computes the
same quantities **analytically** from the ELT's distributions — no simulation, no sampling noise.

### The structure that makes it possible

Within an occurrence of event *e*, losses are driven by one shared shock `z` plus private noise.
**Given `z`, the accounts are independent.** So everything follows from two per-event objects on a
loss grid:

    f_e(x)   = density of the event's portfolio loss
    h_{a,e}(x) = E[L_a | X_e = x] · f_e(x)

with `Σ_a h_{a,e}(x) = x · f_e(x)` by construction. Both are built by convolving the conditionally
independent account severities at each quadrature node and mixing over `z`.

`E[L_a | X_e = x, z]` is computed **exactly** — not approximated by `E[L_a | z]` — via

    E[L_a | X = x, z] = [ (y·p_a) ⊛ p_rest ](x) / p_X(x)

using prefix/suffix products of the account characteristic functions, so no deconvolution is needed.
This matters when `sd_indep` is comparable to `sd_corr`, which it is on real books.

### The two bases

**OEP** conditions on a single occurrence, so it is a direct mixture over which event caused it:

    E[L_a | X = x*] = Σ_e λ_e h_{a,e}(x*) / Σ_e λ_e f_e(x*)

**AEP** conditions on an annual *total*, which is a compound Poisson sum. The Palm decomposition
gives

    E[L_a | S = s*] = Σ_e λ_e (h_{a,e} ⊛ p_S)(s*) / p_S(s*)

where `p_S = IFFT(exp(Σ_e λ_e (FFT(f_e) − 1)))` is the aggregate density. Additivity is exact here
too, because `Σ_e λ_e ((x f_e) ⊛ p_S)(s) = s · p_S(s)` — the compound-Poisson density identity.

### AAL

Needs none of this: `Σ_e rate_e × mean_loss_{a,e}` straight from the ELT, exact and free.

Degenerate legs (zero sd, or mean at/above exposure) are carried as a **point mass** at their
deterministic loss, mirroring the simulator's constant-loss branch — dropping them would silently
zero those accounts' analytic shares.

In [ ]:
# ---- analytic AAL contribution: closed form, no simulation ---------------
aal_analytic = (elt.assign(_c=elt.rate * elt.mean_loss)
                .groupby("accnt_no")["_c"].sum().reindex(accounts).fillna(0.0))
aal_emp = pd.Series(AEP.mean(0), index=accounts)

aal_cmp = pd.DataFrame({"analytic (ELT)": aal_analytic,
                        "empirical (simulated)": aal_emp,
                        "share %": 100 * aal_analytic / aal_analytic.sum(),
                        "MC error %": 100 * (aal_emp / aal_analytic - 1)})
display(aal_cmp.round(3))
print(f"total AAL: analytic {aal_analytic.sum():,.2f} | simulated {aal_emp.sum():,.2f}")
print("use the analytic column -- there is no reason to carry Monte Carlo error on a closed form.")

In [ ]:
# ---- grid and quadrature -------------------------------------------------
from numpy.polynomial.hermite import hermgauss

GRID_N   = 1 << 15                        # cells; raise if the aliasing check below fails
COVER    = 6.0                            # grid spans COVER x the AEP VaR
N_NODES  = 16
MAX_EVTS = None                           # None = all events; else top-N by rate x mean loss

VaR_aep_p = np.sort(Y)[-m_rp]
VaR_oep_p = np.sort(X)[-m_rp]
GRID_H    = COVER * VaR_aep_p / GRID_N
xg        = np.arange(GRID_N) * GRID_H

_nd, _wt = hermgauss(N_NODES)
Z_NODES, Z_W = np.sqrt(2) * _nd, _wt / np.sqrt(np.pi)

_arg_cache = {}
def _arg(a_, b_, E):
    """Phi^-1(BetaCDF(x/E)) on the grid -- depends on the account only, not on z."""
    key = (round(a_, 9), round(b_, 9), round(E, 6))
    if key not in _arg_cache:
        bc = stats.beta.cdf(np.clip((xg + GRID_H / 2) / E, 0.0, 1.0), a_, b_)   # midpoint
        # bins: mass for (x-H/2, x+H/2] sits AT x. Right-endpoint binning biased AAL +1%.
        _arg_cache[key] = stats.norm.ppf(np.clip(bc, 1e-15, 1 - 1e-15))
    return _arg_cache[key]

def _legs(eid):
    out = []
    for r in elt[elt.event_id == eid].itertuples(index=False):
        E = float(r.exposure)
        m_, s_ = r.mean_loss / E, (r.sd_indep + r.sd_corr) / E
        if E <= 0 or not (0 < m_ < 1) or s_ <= 0:
            # the simulator emits these as CONSTANT mean_loss occurrences; mirror them as
            # a point mass, or the account silently gets zero in every analytic share
            out.append((r.accnt_no, None, None, float(r.mean_loss), 0.0, 1.0))
            continue
        a_, b_, _ = beta_params_from_moments(m_, s_)
        w = np.hypot(r.sd_corr, r.sd_indep)
        wc, wi = (0.0, 1.0) if w <= 0 else (r.sd_corr / w, r.sd_indep / w)
        out.append((r.accnt_no, float(a_), float(b_), E, float(wc), float(wi)))
    return out

def event_kernel(eid):
    """f_e(x) and h_{a,e}(x) = E[L_a | X_e = x] f_e(x), with the conditional mean EXACT."""
    legs = _legs(eid)
    if not legs:
        return None
    f_e = np.zeros(GRID_N)
    h   = np.zeros((N_ACC, GRID_N))
    for zj, wj in zip(Z_NODES, Z_W):
        pms, cfs = [], []
        for _, a_, b_, E, wc, wi in legs:
            if a_ is None:                            # degenerate leg: delta at mean_loss
                p = np.zeros(GRID_N); p[min(int(round(E / GRID_H)), GRID_N - 1)] = 1.0
            else:
                cdf = stats.norm.cdf((_arg(a_, b_, E) - wc * zj) / max(wi, 1e-12))
                p   = np.diff(np.concatenate([[0.0], cdf]))
                p   = np.clip(p, 0, None); p[-1] += max(0.0, 1.0 - p.sum())
            pms.append(p); cfs.append(np.fft.fft(p))
        n   = len(cfs)
        pre = [np.ones(GRID_N, complex)]                 # prefix / suffix products avoid
        for c in cfs: pre.append(pre[-1] * c)            # deconvolving to get "all but a"
        suf = [np.ones(GRID_N, complex)]
        for c in reversed(cfs): suf.append(suf[-1] * c)
        suf = suf[::-1]
        p_tot = np.clip(np.real(np.fft.ifft(pre[-1])), 0, None)
        f_e  += wj * p_tot
        for i, (acct, *_rest) in enumerate(legs):
            cf_rest = pre[i] * suf[i + 1]                # everything except account i
            num = np.real(np.fft.ifft(np.fft.fft(xg * pms[i]) * cf_rest))
            h[A_IDX[acct]] += wj * np.clip(num, 0, None)
    return f_e, h

# ---- build kernels, aggregate density, and the Palm numerators -----------
rate_of = elt.groupby("event_id")["rate"].first()
order   = (elt.assign(_c=elt.rate * elt.mean_loss).groupby("event_id")["_c"].sum()
             .sort_values(ascending=False))
evts    = order.index if MAX_EVTS is None else order.index[:MAX_EVTS]
print(f"{len(evts):,} events | covering {100*order.loc[evts].sum()/order.sum():.2f}% of AAL")

logcf = np.zeros(GRID_N, dtype=complex)      # for p_S
H_a   = np.zeros((N_ACC, GRID_N))            # Sum_e lambda_e h_{a,e}
F_x   = np.zeros(GRID_N)                     # Sum_e lambda_e f_e   (OEP denominator)
H_oep = np.zeros((N_ACC, GRID_N))            # Sum_e lambda_e h_{a,e} (OEP numerator)
for eid in evts:
    r = event_kernel(eid)
    if r is None:
        continue
    f_e, h = r
    lam = float(rate_of.loc[eid])
    logcf += lam * (np.fft.fft(f_e) - 1.0)
    H_a   += lam * h
    F_x   += lam * f_e
    H_oep += lam * h

p_S = np.clip(np.real(np.fft.ifft(np.exp(logcf))), 0, None)
p_S /= p_S.sum()

# ---- validation: the analytic aggregate must match the simulated book ----
print(f"p_S mass {p_S.sum():.6f} | analytic AAL {float((p_S*xg).sum()):,.2f} "
      f"vs ELT {aal_analytic.sum():,.2f}")
print(f"aliasing check: mass beyond {COVER:.0f}x VaR = {p_S[-GRID_N//64:].sum():.2e} (want ~0)")
print(f"event-severity mass in the grid's top half = "
      f"{F_x[GRID_N//2:].sum()/max(F_x.sum(), 1e-300):.2e} (circular-wrap guard, want ~0)")

# whole-distribution check: the analytic VaRs must land on the simulated ones to within
# the empirical quantile's own sampling noise (~1-2% at this T). Level-dependent drift
# here means the severity construction is wrong - this is the diagnostic that catches it.
var_ana_aep = float(xg[np.searchsorted(np.cumsum(p_S), ALPHA_RP)])
tail_rate   = F_x.sum() - np.cumsum(F_x)                # sum_e lambda_e P(X_e > x)
var_ana_oep = float(xg[np.searchsorted(-(1.0 - np.exp(-tail_rate)), -(1 - ALPHA_RP))])
print(f"analytic vs empirical VaR at RP {RP}: "
      f"AEP {var_ana_aep:,.1f}/{VaR_aep_p:,.1f} = {var_ana_aep/VaR_aep_p:.4f} | "
      f"OEP {var_ana_oep:,.1f}/{VaR_oep_p:,.1f} = {var_ana_oep/VaR_oep_p:.4f}")

In [ ]:
# ---- OEP: E[L_a | X = VaR_oep] -------------------------------------------
i_oep = int(np.clip(np.searchsorted(xg, VaR_oep_p), 1, GRID_N - 1))
num_o = H_oep[:, i_oep]
ana_oep = pd.Series(num_o / max(F_x[i_oep], 1e-300), index=accounts)
ana_oep = ana_oep / ana_oep.sum() * VaR_oep_p            # enforce additivity on the grid

# ---- AEP: E[L_a | S = VaR_aep] via the Palm decomposition ----------------
i_aep = int(np.clip(np.searchsorted(xg, VaR_aep_p), 1, GRID_N - 1))
cf_pS = np.fft.fft(p_S)
num_a = np.array([np.real(np.fft.ifft(np.fft.fft(H_a[k]) * cf_pS))[i_aep]
                  for k in range(N_ACC)])
ana_aep = pd.Series(num_a / max(p_S[i_aep], 1e-300), index=accounts)
ana_aep = ana_aep / ana_aep.sum() * VaR_aep_p

display(pd.DataFrame({"AEP  E[L|S=VaR]": ana_aep, "AEP share %": 100*ana_aep/ana_aep.sum(),
                      "OEP  E[L|X=VaR]": ana_oep, "OEP share %": 100*ana_oep/ana_oep.sum()})
        .sort_values("AEP share %", ascending=False).round(2))
print(f"AEP sums to {ana_aep.sum():,.2f} = VaR {VaR_aep_p:,.2f} | "
      f"OEP sums to {ana_oep.sum():,.2f} = VaR {VaR_oep_p:,.2f}")

## 9 · All methods, both bases

Every column is a set of proportions rescaled to the same VaR total, so they are directly
comparable. What differs is only **where the proportions come from**:

- **co-TVaR** — average over every year beyond VaR. Lowest noise, biased toward the deep tail.
- **win K** — average over years ranked `m−K` to `m+K`. Narrow is closer to the VaR point but
  noisier; wide reaches into the body.
- **analytic** — `E[L_a | loss = VaR]` from the ELT's distributions. No sampling noise at all, but
  it assumes the Beta/copula/Poisson model and a finite grid.

The analytic column is what the empirical ones are estimating, so read it as the reference: an
empirical estimator close to it is validated, and a large gap says either the tail is too thin to
estimate from or the model assumption is doing real work.

On the OEP basis every empirical column reads the account's loss **on the year's biggest
occurrence** — the additive object of §5 and the analytic column's estimand — not the account's own
annual max, which inflates small diversified accounts. Widths reaching past the retained years also
pick up share-reconstruction error in the body (measured ~1–4% at win1000 here; check on your book).

In [ ]:
def emp_props(A, ord_, K, m_):
    p = A[ord_[:m_]].mean(0) if K is None else \
        A[ord_[max(0, m_-1-K): m_-1+K+1]].mean(0)
    return p / p.sum()

ord_A = np.argsort(-Y, kind="stable")
ord_O = np.argsort(-X, kind="stable")
WIDTHS = [50, 200, 1000]

def basis_table(A, ord_, VaR_, ana):
    cols = {f"win{K}": emp_props(A, ord_, K, m_rp) * VaR_ for K in WIDTHS}
    cols["co-TVaR"]  = emp_props(A, ord_, None, m_rp) * VaR_
    cols["analytic"] = ana.to_numpy()
    return pd.DataFrame(cols, index=accounts)

aep_tbl = basis_table(AEP, ord_A, VaR_aep_p, ana_aep)
oep_tbl = basis_table(BIG, ord_O, VaR_oep_p, ana_oep)   # loss on the year's biggest
# occurrence - the additive object of section 5 and the same estimand as the analytic
# column. The annual-max matrix would inflate small diversified accounts 10-20%.

print(f"=== AEP basis, RP {RP} (VaR {VaR_aep_p:,.2f}) ===")
display((100 * aep_tbl / VaR_aep_p).round(2))
print(f"=== OEP basis, RP {RP} (VaR {VaR_oep_p:,.2f}) ===")
display((100 * oep_tbl / VaR_oep_p).round(2))
print("columns are capital share %; every column sums to 100.")

In [ ]:
# ---- how far is each empirical method from the analytic reference? -------
rows = []
for lab, tbl in [("AEP", aep_tbl), ("OEP", oep_tbl)]:
    for c in tbl.columns:
        if c == "analytic":
            continue
        d = (tbl[c] - tbl["analytic"]).abs() / tbl["analytic"].replace(0, np.nan) * 100
        rows.append({"basis": lab, "method": c,
                     "median gap vs analytic %": float(d.median()),
                     "max gap %": float(d.max()),
                     "worst account": d.idxmax()})
display(pd.DataFrame(rows).set_index(["basis", "method"]).round(2))

# ---- and what each empirical method costs in sampling noise -------------
rb, B = np.random.default_rng(5), 150
noise = []
for lab, A, Yv, VaR_ in [("AEP", AEP, Y, VaR_aep_p), ("OEP", BIG, X, VaR_oep_p)]:
    for K in WIDTHS + [None]:
        bs = np.empty((B, N_ACC))
        for b in range(B):
            idx = rb.integers(0, T_YEARS, T_YEARS)
            bs[b] = emp_props(A[idx], np.argsort(-Yv[idx], kind="stable"), K, m_rp) * VaR_
        base = emp_props(A, np.argsort(-Yv, kind="stable"), K, m_rp) * VaR_
        noise.append({"basis": lab, "method": "co-TVaR" if K is None else f"win{K}",
                      "median SE %": float(np.nanmedian(100 * bs.std(0) / base))})
display(pd.DataFrame(noise).set_index(["basis", "method"]).round(2))
print()
print("analytic has NO sampling SE by construction; its error is model + grid instead.")
print("pick the empirical width whose gap-to-analytic and SE are both acceptable, or use the")
print("analytic split directly once you are satisfied the two agree.")

### Reading the two tables together

- **Gap vs analytic** is bias. It tells you how far a given window sits from the quantity it is
  meant to estimate.
- **Median SE** is noise. It tells you how much that column would move on a rerun.

A method is usable when both are small. If every empirical method sits close to analytic, take
co-TVaR for its stability. If they diverge as the window narrows, that divergence is the deep-tail
bias in co-TVaR made visible — and the analytic column is the tiebreak.

**Grid check before trusting the analytic column:** halve `GRID_H` (raise `GRID_N`) and confirm the
shares are stable to a fraction of a point. The aliasing diagnostic in §8 should stay near zero.

**On a large catalogue** the kernel loop is the cost — it is `O(events × nodes × footprint)` FFTs.
Set `MAX_EVTS` to the top few hundred events by `rate × mean_loss` and read the coverage line; the
omitted events contribute negligibly to the tail but the coverage number should be reported.

## 10 · Benchmark — leave-one-out marginals

The ground-truth question behind every allocation: **what actually happens to the RP-200 loss and
the AAL if an account is removed?** Remove each account from the book, recompute both, difference
against the full book.

Two things make this exact and cheap:

- **No resimulation is needed.** The shared shock is a property of the *occurrence*, not of which
  accounts exist, so under common random numbers the without-account book is **exactly the full
  simulation minus that account's rows**. Literally re-running the engine on the reduced ELT with
  the same seed would shift the RNG stream (removed legs stop consuming draws) and resample every
  *remaining* account — the demo below measures that contamination. This also means the benchmark
  works unchanged in `PLT` mode, where no engine exists.
- **ΔAAL is the account's own AAL identically** — means carry no diversification — so the AAL
  benchmark reduces to the §8 analytic-vs-simulated comparison.

ΔVaR is different: the columns of §9 sum to VaR by construction, but leave-one-out differences do
**not** — the gap is diversification, and per account it is the locality principle made numeric.
An allocation prices the account *at today's mix*; the LOO number prices *removing it entirely*,
a large move. Expect them to agree for small accounts and diverge for large ones.

In [ ]:
# --- the benchmark: exact CRN leave-one-out from the one simulation --------
top_m = lambda v: np.partition(v, -m_rp)[-m_rp]          # m_rp-th largest, same convention
loo = pd.DataFrame({
    "LOO dVaR": [VaR_aep_p - top_m(Y - AEP_T[:, i]) for i in range(N_ACC)],
    "LOO dAAL": AEP_T.mean(0)}, index=accounts)

# coupled bootstrap SE of the VaR difference (resample years, difference within replicate)
rb, B = np.random.default_rng(11), 200
bs = np.empty((B, N_ACC))
for b in range(B):
    idx = rb.integers(0, T_YEARS, T_YEARS)
    Yb, Ab = Y[idx], AEP_T[idx]
    vb = top_m(Yb)
    bs[b] = [vb - top_m(Yb - Ab[:, i]) for i in range(N_ACC)]
loo["SE"] = bs.std(0)

print(f"sum of LOO dVaR {loo['LOO dVaR'].sum():,.2f} vs portfolio VaR {VaR_aep_p:,.2f} -> "
      f"diversification gap {100*(1 - loo['LOO dVaR'].sum()/VaR_aep_p):+.1f}%")
print("(allocation columns sum to VaR by construction; LOO differences never do)")

# --- why not literally resimulate: measure the stream-shift contamination --
if INPUT_MODE == "ELT":
    a_demo  = accounts[int(np.argmax(AEP_T.mean(0)))]
    plt_wo  = elt_to_account_plt(elt[elt.accnt_no != a_demo], n_years=T_YEARS,
                                 seed=SEED, verbose=False)
    plt_wo["year_id"] -= 1
    Y_naive = np.bincount(plt_wo.year_id, weights=plt_wo.loss, minlength=T_YEARS)
    Y_crn   = Y - AEP_T[:, A_IDX[a_demo]]
    d_naive = VaR_aep_p - top_m(Y_naive)
    d_crn   = float(loo.loc[a_demo, "LOO dVaR"])
    print(f"\nnaive same-seed resimulation, {a_demo}: remaining-book years differ by up to "
          f"{np.abs(Y_naive - Y_crn).max():,.1f}")
    print(f"  dVaR naive {d_naive:,.2f} vs exact-CRN {d_crn:,.2f} "
          f"({(d_naive-d_crn)/loo.loc[a_demo,'SE']:+.1f} SE of spurious drift)")

In [ ]:
# --- the comparison: every method against the benchmark, RP 200 AEP --------
bench = pd.DataFrame({
    "benchmark LOO dVaR": loo["LOO dVaR"], "SE": loo["SE"],
    "co-TVaR x VaR (S6)": aep_tbl["co-TVaR"],
    "win200":             aep_tbl["win200"],
    "win50":              aep_tbl["win50"],
    "analytic E[L|S=VaR]": ana_aep,
    "raw co-VaR (1 yr)":  pd.Series(co_var, index=accounts)})
display(bench.round(1))

meth = [c for c in bench.columns if c not in ("benchmark LOO dVaR", "SE")]
z    = bench[meth].sub(bench["benchmark LOO dVaR"], axis=0).div(bench["SE"], axis=0)
summ = pd.DataFrame({
    "median |gap| %": (bench[meth].sub(bench["benchmark LOO dVaR"], axis=0).abs()
                       .div(bench["benchmark LOO dVaR"], axis=0).median() * 100),
    "accounts within 2 SE": (z.abs() < 2).sum(),
    "sums to": bench[meth].sum()})
display(summ.round(2))

# AAL benchmark closes the loop with section 8
aal_bench = pd.DataFrame({"LOO dAAL": loo["LOO dAAL"],
                          "analytic (ELT)": aal_analytic,
                          "MC error %": 100*(loo["LOO dAAL"]/aal_analytic - 1)})
display(aal_bench.round(3))
print("LOO dAAL is the account's own simulated AAL identically; its gap to the analytic")
print("column is Monte Carlo error only. For capital, the benchmark says: methods that")
print("agree with LOO within its SE are validated for small accounts; a systematic gap on")
print("the largest accounts is the discrete-removal vs at-the-margin distinction (S6/S29),")
print("not an estimator failure - do not tune a window to chase it.")

### Reading the benchmark

For each account, `benchmark LOO dVaR ± SE` is what removal actually does. The method columns are
allocations of today's VaR. Three regimes to expect: small accounts — everything agrees within SE,
and the benchmark validates the whole §9 stack; mid accounts — the analytic and window
columns track the benchmark best (both estimate the local gradient); the largest accounts show the
widest gaps, in whichever direction the book dictates — the sign depends on how the account's tail
contribution varies along the removal path, so measure it, never assume it. That gap is
*information*, not error: it is the difference between
"price this renewal" (marginal) and "what if we lose the whole relationship" (LOO), and both belong
in the pack with their SEs.

## Switching input mode

`INPUT_MODE = "PLT"` skips §0 entirely — point cell §1 at your own `read_parquet` calls and every
other cell runs unchanged. `INPUT_MODE = "ELT"` runs the engine and derives the PLTs from it.

**If you scale up the ELT path**, check `elt.groupby("event_id").size().describe()` first. The engine
loops events in Python and vectorises across occurrences within each, so a catalogue of tens of
thousands of events with narrow footprints runs slower per row than a smaller catalogue with wide
ones. If that bites, parallelise the **event** loop — never the account loop inside it, whose members
must consume the same `z_shared`.